In [0]:
%skip
PERFORMANCE LOJAS
LOJAS COM MAIS DEVOLUÇÕES
LOJAS COM MAIS VENDAS POR DIA, MES E ANO
LOJA COM MAIS CLIENTES
DADOS ACIMA POR LOJA FISICA E VIRTUAL
INVESTIMENTO EM MARKETING ETC

In [0]:
import pandas as pd

# Lê os arquivos
df1 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Vendas.csv')
# df2 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Clientes.csv')
df3 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Lojas.csv')
df4 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Produtos.csv')
#df5 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Historico_Vendas.csv')
df6 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Campanhas.csv')
df7 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Custos_Operacionais.csv')
df8 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Devolucoes.csv')

In [0]:
%skip
df1 (vendas)
+ df3 (lojas)
+ df4 (produtos)
+ df8 (devolucoes)
+ df6 (campanhas)
+ df7 (custos)

In [0]:
df1 = df1.rename(columns={
    'Venda_ID': 'id_venda',
    'Loja_ID': 'id_loja',
    'Produto_ID': 'id_produto',
    'Cliente_ID': 'id_cliente',
    'Colaborador_ID': 'id_colaborador',
    'Quantidade': 'quantidade',
    'Preço Unitário': 'preco_unitario',
    'Data da Venda': 'data_venda',
    'Canal de Venda': 'canal_venda'
})

df1['quantidade'] = pd.to_numeric(df1['quantidade'], errors='coerce')
df1['preco_unitario'] = pd.to_numeric(df1['preco_unitario'], errors='coerce')
df1 = df1.dropna(subset=['quantidade', 'preco_unitario'])
df1['data_venda'] = pd.to_datetime(df1['data_venda'], errors='coerce')
df_vendas = df1

In [0]:
spark_df_vendas = spark.createDataFrame(df_vendas)
spark_df_vendas.createOrReplaceTempView("temp_vendas")

In [0]:
%skip
df2 = df2.rename(columns={
    'Cliente_ID': 'id_cliente',
    'Nome': 'nome_cliente',
    'Idade': 'idade_cliente',
    'Género': 'genero',
    'Cidade': 'cidade_cliente'
})
df_clientes = df2[['id_cliente', 'nome_cliente', 'idade_cliente', 'genero', 'cidade_cliente']]

In [0]:
%skip
spark_df_clientes = spark.createDataFrame(df_clientes)
spark_df_clientes.createOrReplaceTempView("temp_clientes")

In [0]:
df3 = df3.rename(columns={
    'Loja_ID': 'id_loja',
    'Nome': 'nome_loja',
    'Região': 'regiao_loja',
    'Cidade': 'cidade_loja',
    'Tipo': 'tipo_loja'
})
df_lojas = df3[['id_loja', 'nome_loja', 'regiao_loja', 'cidade_loja', 'tipo_loja']]

In [0]:
spark_df_lojas = spark.createDataFrame(df_lojas)
spark_df_lojas.createOrReplaceTempView("temp_lojas")

In [0]:
%sql
select *
from temp_lojas

In [0]:
df4 = df4.rename(columns={
    'Produto_ID': 'id_produto',
    'Nome': 'nome_produto',
    'Categoria': 'categoria',
    'Cor': 'cor',
    'Descrição': 'descricao',
    'Tamanho': 'tamanho',
    'Preço': 'preco',
    'Custo_Aquisição': 'custo_aquisicao',
    'Imagem': 'imagem'
})
df_produtos= df4[['id_produto', 'nome_produto', 'categoria']]
df_produtos.head()

In [0]:
import pandas as pd

# converter para pandas
df_produtos= df4[['id_produto', 'nome_produto', 'categoria']]

roupas = [
"Camiseta Nike","Camiseta Adidas","Camiseta Puma","Camisa Polo","Camisa Social",
"Jaqueta Jeans","Jaqueta Corta Vento","Moletom Nike","Moletom Adidas",
"Calça Jeans","Calça Moletom","Calça Cargo","Bermuda Jeans","Bermuda Esportiva",
"Vestido Casual","Vestido Longo","Blusa Feminina","Blusa Cropped",
"Saia Jeans","Saia Midi","Regata Esportiva","Regata Nike","Regata Adidas"
]

calcados = [
"Tenis Nike Air","Tenis Adidas Ultraboost","Tenis Puma Runner","Tenis Vans Old Skool",
"Tenis Converse All Star","Bota Timberland","Bota Coturno","Sapato Social",
"Sapato Oxford","Sandalia Feminina","Sandalia Rasteira","Chinelo Havaianas",
"Chinelo Slide Nike","Tenis Nike SB","Tenis Adidas Runfalcon","Tenis Mizuno Wave",
"Tenis Asics Gel","Tenis New Balance","Tenis Olympikus","Tenis Fila"
]

i_roupa = 0
i_calcado = 0

for i,row in df_produtos.iterrows():
    
    if row["categoria"] == "Roupas":
        df_produtos.at[i,"nome_produto"] = roupas[i_roupa % len(roupas)]
        i_roupa += 1
        
    else:
        df_produtos.at[i,"nome_produto"] = calcados[i_calcado % len(calcados)]
        i_calcado += 1

In [0]:
spark_df_produtos = spark.createDataFrame(df_produtos)
spark_df_produtos.createOrReplaceTempView("temp_produtos")

In [0]:
df6 = df6.rename(columns={
    'Campanha_ID': 'id_campanha',
    'Loja_ID': 'id_loja',
    'Nome': 'nome',
    'Canal': 'canal',
    'Investimento': 'investimento',
    'Vendas Geradas': 'vendas_geradas',
    'Data de Início': 'dt_inicio',
    'Data de Fim': 'dt_fim'
})
df_campanhas= df6
#[['id_produto', 'nome_produto', 'categoria']]
df_campanhas.head()

In [0]:
spark_df_campanhas = spark.createDataFrame(df_campanhas)
spark_df_campanhas.createOrReplaceTempView("temp_campanhas")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_campanhas2 AS
SELECT 
    c.id_campanha,
    d.id_loja_corrigido AS id_loja,
    c.nome,
    c.canal,
    c.investimento,
    c.vendas_geradas,
    c.dt_inicio,
    c.dt_fim
FROM temp_campanhas c
LEFT JOIN databricks_database.silver.depara_lojas d
    ON c.id_loja = d.id_loja

In [0]:
%sql
--CREATE OR REPLACE TABLE databricks_database.silver.depara_lojas AS
SELECT 
    id_loja,
    (ABS(HASH(id_loja)) % 5) + 1 AS id_loja_corrigido
FROM temp_campanhas
group by all

In [0]:
df7 = df7.rename(columns={
    'Custo_ID': 'id_custo',
    'Loja_ID': 'id_loja',
    'Tipo de Custo': 'tipo_custo',
    'Valor Mensal': 'vlr_mensal',
    'Data': 'datao'
})
df_custos= df7
#[['id_produto', 'nome_produto', 'categoria']]
df_custos.head()

In [0]:
spark_df_custos = spark.createDataFrame(df_custos)
spark_df_custos.createOrReplaceTempView("temp_custos")

In [0]:
%sql
select min(id_loja) as minimo
, max(id_loja) as max
from temp_custos
group by all

In [0]:
df8 = df8.rename(columns={
    'Devolução_ID': 'id_devolucao',
    'Venda_ID': 'id_venda',
    'Produto_ID': 'id_produto',
    'Cliente_ID': 'id_cliente',
    'Quantidade': 'quantidade',
    'Motivo da Devolução': 'motivo_devolucao',
    'Data da Devolução':'dt_devolucao'
})
df_devolucao= df8
#[['id_produto', 'nome_produto', 'categoria']]
df_devolucao.head()

In [0]:
spark_df_devolucao = spark.createDataFrame(df_devolucao)
spark_df_devolucao.createOrReplaceTempView("temp_devolucao")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW final_vendas AS

WITH devolucoes_agg AS (
    SELECT 
        id_venda,
        SUM(quantidade) AS qtd_devolvida
    FROM temp_devolucao
    GROUP BY id_venda
)

SELECT 
    -- IDENTIFICAÇÃO
    v.id_venda,
    v.data_venda,

    -- TEMPO
    YEAR(v.data_venda) AS ano,
    MONTH(v.data_venda) AS mes,
    DAY(v.data_venda) AS dia,

    -- LOJA
    v.id_loja,
    l.nome_loja,
    l.tipo_loja,
    l.regiao_loja,
    l.cidade_loja,

    -- CLIENTE
    v.id_cliente,

    -- PRODUTO
    v.id_produto,
    p.nome_produto,
    p.categoria,

    -- MÉTRICAS
    v.quantidade,
    v.preco_unitario,
    (v.quantidade * v.preco_unitario) AS valor_total,

    v.canal_venda,

    -- DEVOLUÇÃO
    CASE 
        WHEN d.id_venda IS NOT NULL THEN 1 
        ELSE 0 
    END AS teve_devolucao,

    d.qtd_devolvida

FROM temp_vendas v

LEFT JOIN temp_lojas l 
    ON v.id_loja = l.id_loja

LEFT JOIN temp_produtos p 
    ON v.id_produto = p.id_produto

LEFT JOIN devolucoes_agg d 
    ON v.id_venda = d.id_venda

In [0]:
%sql

CREATE OR REPLACE TABLE databricks_database.gold.visao_performance_lojas AS 
  select *
  from final_vendas

In [0]:
%sql
Select sum(valor_total)
from final_vendas
group by all

In [0]:
%sql
Select sum(receita_total)
from databricks_database.gold.visao_vendas_dashboard
group by all

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW final_lojas AS

WITH campanhas_agg AS (
    SELECT 
        id_loja,
        YEAR(dt_inicio) AS ano,
        MONTH(dt_inicio) AS mes,
        SUM(investimento) AS investimento_total
    FROM temp_campanhas2
    GROUP BY id_loja, YEAR(dt_inicio), MONTH(dt_inicio)
),

custos_agg AS (
    SELECT 
        id_loja,
        YEAR(datao) AS ano,
        MONTH(datao) AS mes,
        SUM(vlr_mensal) AS custo_operacional
    FROM temp_custos
    GROUP BY id_loja, YEAR(datao), MONTH(datao)
)

SELECT 
    COALESCE(c.id_loja, co.id_loja) AS id_loja,
    COALESCE(c.ano, co.ano) AS ano,
    COALESCE(c.mes, co.mes) AS mes,
    c.investimento_total,
    co.custo_operacional
FROM campanhas_agg c
FULL OUTER JOIN custos_agg co
    ON c.id_loja = co.id_loja
    AND c.ano = co.ano
    AND c.mes = co.mes

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_database.gold.visao_performance_lojas_custos AS
  select *
  from final_lojas

In [0]:
%sql
select * 
from final_lojas

In [0]:
%sql 
--performance
SELECT 
    id_loja,
    nome_loja,
    SUM(valor_total) AS faturamento,
    COUNT(DISTINCT id_venda) AS total_vendas,
    COUNT(DISTINCT id_cliente) AS total_clientes
FROM final_vendas
GROUP BY id_loja, nome_loja
ORDER BY faturamento DESC

In [0]:
%sql
--devolucoes lojas
SELECT 
    id_loja,
    nome_loja,
    --SUM(CASE WHEN teve_devolucao = 1 THEN 1 ELSE 0 END) AS qtd_devolucoes
    SUM(CASE WHEN teve_devolucao = 1 THEN 1 ELSE 0 END) AS qtd_devolucoes,
    SUM(qtd_devolvida) AS total_itens_devolvidos
FROM final_vendas
GROUP BY id_loja, nome_loja
ORDER BY qtd_devolucoes DESC

In [0]:
%sql
SELECT 
    ano,
    mes,
    SUM(valor_total) AS faturamento
FROM final_vendas
GROUP BY ano, mes
ORDER BY ano, mes

In [0]:
%sql
SELECT 
    id_loja,
    nome_loja,
    COUNT(DISTINCT id_cliente) AS total_clientes
FROM final_vendas
GROUP BY id_loja, nome_loja
ORDER BY total_clientes DESC

In [0]:
%sql
SELECT 
    tipo_loja,
    SUM(valor_total) AS faturamento,
    COUNT(DISTINCT id_cliente) AS clientes
FROM final_vendas
GROUP BY tipo_loja

In [0]:
%sql
SELECT 
    v.id_loja,
    v.ano,
    --v.mes,
    SUM(v.valor_total) AS faturamento,
    MAX(lm.investimento_total) AS investimento
FROM final_vendas v
LEFT JOIN final_lojas lm
    ON v.id_loja = lm.id_loja
    AND v.ano = lm.ano
    --AND v.mes = lm.mes
GROUP BY v.id_loja, v.ano--, v.mes
ORDER BY faturamento DESC

In [0]:
%sql
SELECT 
    v.id_loja,
    v.ano,
    v.mes,
    SUM(v.valor_total) AS faturamento,
    MAX(lm.investimento_total) AS investimento,
    
    CASE 
        WHEN MAX(lm.investimento_total) > 0 
        THEN SUM(v.valor_total) / MAX(lm.investimento_total)
        ELSE NULL
    END AS roi
FROM final_vendas v
LEFT JOIN final_lojas as lm
    ON v.id_loja = lm.id_loja
    AND v.ano = lm.ano
    AND v.mes = lm.mes
GROUP BY v.id_loja, v.ano, v.mes